[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/06_human_in_the_loop.ipynb) [![GitHub](https://img.shields.io/badge/GitHub-仓库主页-181717?logo=github)](https://github.com/Luca5702/langgraph-study)

**📚 课程导航：** [01](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/01_hello_graph.ipynb) · [02](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/02_state_reducers.ipynb) · [03](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/03_control_flow.ipynb) · [04](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/04_agent_tools.ipynb) · [05](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/05_persistence_memory.ipynb) · **[06](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/06_human_in_the_loop.ipynb)** · [07](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/07_time_travel.ipynb) · [08](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/08_streaming.ipynb) · [09](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/09_subgraphs.ipynb) · [10](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/10_functional_api.ipynb) · [11](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/11_production.ipynb)

[⬅️ 上一课：第 5 课：持久化与记忆（Checkpointer + Store）](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/05_persistence_memory.ipynb) ｜ [➡️ 下一课：第 7 课：时间旅行（Replay & Fork）](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/07_time_travel.ipynb)

---

# 第 6 课：Human-in-the-loop —— interrupt() 与 Command(resume=...)

* 在节点内调用 interrupt(payload)：图暂停，payload 返回给调用方（result["__interrupt__"]）。
* 调用方用 graph.invoke(Command(resume=value), config) 恢复；
  恢复时该节点会"从头重新执行"，这一次 interrupt(...) 直接返回 value。
* 前提：必须有 checkpointer + thread_id。
* 规则（重要！）：
  1. 不要用 try/except 包住 interrupt（它靠抛异常实现暂停）
  2. 同一节点内多个 interrupt 的顺序不能变（按索引匹配 resume 值）
  3. interrupt 之前的副作用必须幂等（因为节点会重跑）

In [1]:
#@title 🔧 第 0 步：初始化环境（在 Colab 中点击左侧 ▶ 即可自动安装依赖，本地运行自动跳过） { display-mode: "form" }
import importlib.util, subprocess, sys
if importlib.util.find_spec('langgraph') is None:
    print('⏳ 正在安装 LangGraph 环境（约 10 秒）...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'langgraph', 'langchain-core', 'langgraph-checkpoint-sqlite'])
    print('✅ 安装完成！')
else:
    import importlib.metadata as m
    print(f'✅ 环境已就绪：langgraph={m.version("langgraph")}')


✅ 环境已就绪：langgraph=1.2.22


In [2]:
from typing_extensions import Literal, TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt


class State(TypedDict):
    action: str
    status: str


def plan(state: State):
    print("  [plan] 计划执行:", state["action"])
    return {}


def human_review(state: State) -> Command[Literal["execute", "__end__"]]:
    print("  [human_review] 节点开始执行（恢复时会再次打印这行！）")
    decision = interrupt({"question": "是否批准该操作？", "action": state["action"]})
    # ↑ 第一次运行到这里暂停；恢复后 decision = resume 传入的值
    if decision.get("approved"):
        new_action = decision.get("edited_action", state["action"])
        return Command(goto="execute", update={"action": new_action})
    return Command(goto=END, update={"status": "❌ 被拒绝"})


def execute(state: State):
    return {"status": f"✅ 已执行：{state['action']}"}


graph = (
    StateGraph(State)
    .add_node(plan).add_node(human_review).add_node(execute)
    .add_edge(START, "plan").add_edge("plan", "human_review").add_edge("execute", END)
    .compile(checkpointer=InMemorySaver())
)

## ▶️ 运行实验

In [3]:
cfg = {"configurable": {"thread_id": "review-1"}}
print("== 第一次运行 ==")
result = graph.invoke({"action": "删除生产数据库 users 表", "status": ""}, cfg)
print("暂停！__interrupt__ =", result["__interrupt__"])
print("当前 next =", graph.get_state(cfg).next)

print("\n== 人类修改后批准 ==")
result = graph.invoke(Command(resume={"approved": True, "edited_action": "备份 users 表"}), cfg)
print("结果:", result)

print("\n== 另一个线程：拒绝 ==")
cfg2 = {"configurable": {"thread_id": "review-2"}}
graph.invoke({"action": "给所有用户发垃圾邮件", "status": ""}, cfg2)
print("结果:", graph.invoke(Command(resume={"approved": False}), cfg2))

== 第一次运行 ==
  [plan] 计划执行: 删除生产数据库 users 表
  [human_review] 节点开始执行（恢复时会再次打印这行！）
暂停！__interrupt__ = [Interrupt(value={'question': '是否批准该操作？', 'action': '删除生产数据库 users 表'}, id='b8e84074550dee96fd6dc1612dff1181', response_schema=None)]
当前 next = ('human_review',)

== 人类修改后批准 ==
  [human_review] 节点开始执行（恢复时会再次打印这行！）
结果: {'action': '备份 users 表', 'status': '✅ 已执行：备份 users 表'}

== 另一个线程：拒绝 ==
  [plan] 计划执行: 给所有用户发垃圾邮件
  [human_review] 节点开始执行（恢复时会再次打印这行！）
  [human_review] 节点开始执行（恢复时会再次打印这行！）
结果: {'action': '给所有用户发垃圾邮件', 'status': '❌ 被拒绝'}


## ✏️ 练习：
 1. 在一个节点里连续调用两次 interrupt（先问姓名再问年龄），观察需要 resume 几次。
 2. 把 interrupt 放到工具函数内部（工具里也能用！），实现"调用敏感工具前需审批"。
 3. 阅读 docs/src/oss/langgraph/interrupts.mdx 中 "Rules of interrupts"。

In [4]:
# 在这里写你的练习代码 👇


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/06_human_in_the_loop.ipynb) [![GitHub](https://img.shields.io/badge/GitHub-仓库主页-181717?logo=github)](https://github.com/Luca5702/langgraph-study)

**📚 课程导航：** [01](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/01_hello_graph.ipynb) · [02](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/02_state_reducers.ipynb) · [03](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/03_control_flow.ipynb) · [04](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/04_agent_tools.ipynb) · [05](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/05_persistence_memory.ipynb) · **[06](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/06_human_in_the_loop.ipynb)** · [07](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/07_time_travel.ipynb) · [08](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/08_streaming.ipynb) · [09](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/09_subgraphs.ipynb) · [10](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/10_functional_api.ipynb) · [11](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/11_production.ipynb)

[⬅️ 上一课：第 5 课：持久化与记忆（Checkpointer + Store）](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/05_persistence_memory.ipynb) ｜ [➡️ 下一课：第 7 课：时间旅行（Replay & Fork）](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/07_time_travel.ipynb)

---